# **XLM-R ZERO SHOT ENGLISH CLASSIFICATION**

In [ ]:
# ============================================================
# BLOCK 0 — INSTALL
# ============================================================
!pip install -q transformers scikit-learn safetensors matplotlib


# ============================================================
# BLOCK 1 — IMPORTS
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)
from sklearn.preprocessing import label_binarize
from transformers import AutoTokenizer, AutoModel

print("Imports done.")


# ============================================================
# BLOCK 2 — REPRODUCIBILITY
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seeds set | Device: {DEVICE}")


# ============================================================
# BLOCK 3 — LOAD + CLEAN DATA
# ============================================================
DATA_PATH = "final_combined.csv"
for candidate in [
    DATA_PATH,
    "/content/final_combined.csv",
    "/mnt/data/final_combined.csv",
]:
    if os.path.exists(candidate):
        DATA_PATH = candidate
        break

df = None
for encoding in ["utf-8", "latin-1", "cp1252", "utf-8-sig"]:
    try:
        df = pd.read_csv(DATA_PATH, encoding=encoding)
        print(f"Loaded with encoding: {encoding}")
        break
    except UnicodeDecodeError:
        print(f"Failed with encoding: {encoding}")
        continue

if df is None:
    raise ValueError("Could not load CSV with any encoding.")

required_cols = ["sentence", "label", "distortion", "language"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

df = df.dropna(subset=required_cols).copy()
df["sentence"]   = df["sentence"].astype(str).str.strip()
df["label"]      = df["label"].astype(int)
df["distortion"] = df["distortion"].astype(str).str.strip()
df["language"]   = df["language"].astype(str).str.strip()

before = len(df)
df = df.drop_duplicates(subset=["sentence"]).copy()
df = df[df["sentence"].str.len() > 0].copy()
df = df.reset_index(drop=True)

print(f"Removed {before - len(df)} duplicates. Dataset size: {len(df)}")
print("Label distribution:\n", df["label"].value_counts())
print("Language distribution:\n", df["language"].value_counts())
print("Top distortions:\n", df["distortion"].value_counts().head(10))

df["length"] = df["sentence"].str.split().str.len()
print("\nLength stats by label:")
print(df.groupby("label")["length"].describe()[["mean", "50%", "min", "max"]])
df = df.drop(columns=["length"])

print("Data loaded.")


# ============================================================
# BLOCK 4 — TOKENIZER + PRETRAINED MODEL
# ============================================================
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.eval()

print(f"Loaded pretrained encoder: {MODEL_NAME}")


# ============================================================
# BLOCK 5 — HELPER: MEAN POOLING
# ============================================================
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


# ============================================================
# BLOCK 6 — HELPER: EMBED TEXTS
# ============================================================
@torch.no_grad()
def embed_texts(texts, tokenizer, model, batch_size=16, max_len=128, device="cpu"):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        outputs = model(**enc)
        emb = mean_pooling(outputs.last_hidden_state, enc["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu())

    return torch.cat(all_embeddings, dim=0)


print("Embedding helpers ready.")


# ============================================================
# BLOCK 7 — HELPER: PROTOTYPE CLASSIFIER
# ============================================================
def build_prototypes(embeddings, labels, num_classes):
    prototypes = []
    labels = np.array(labels)

    for c in range(num_classes):
        class_embs = embeddings[labels == c]
        if len(class_embs) == 0:
            raise ValueError(f"No samples found for class {c}")
        proto = class_embs.mean(dim=0)
        proto = F.normalize(proto, p=2, dim=0)
        prototypes.append(proto)

    return torch.stack(prototypes, dim=0)


def predict_with_prototypes(val_embeddings, prototypes):
    sims = torch.matmul(val_embeddings, prototypes.T)   # cosine similarity
    preds = sims.argmax(dim=1).numpy()
    return preds, sims.numpy()


print("Prototype classifier ready.")


# ============================================================
# BLOCK 8 — METRICS
# ============================================================
def compute_metrics_from_preds(y_true, y_pred, average="weighted"):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=average, zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return acc, precision, recall, f1

print("Metrics ready.")


# ============================================================
# BLOCK 9 — SANITY CHECKS
# ============================================================
print("\n========== SANITY CHECKS ==========")
print("Null sentences    :", df["sentence"].isnull().sum())
print("Remaining dupes   :", df["sentence"].duplicated().sum())
bin_counts = df["label"].value_counts()
print(f"Class ratio       : {bin_counts.max()/bin_counts.min():.2f}:1")
dist_counts = df[df["label"] == 1]["distortion"].value_counts()
print("Distortion counts :\n", dist_counts)
print("====================================\n")


# ============================================================
# BLOCK 10 — BINARY ZERO-SHOT PREP
# ============================================================
bin_df = df.copy()
bin_df["stratify_key"] = (
    bin_df["label"].astype(str) + "_" + bin_df["language"].astype(str)
)

valid_keys = bin_df["stratify_key"].value_counts()
bin_df = bin_df[bin_df["stratify_key"].isin(valid_keys[valid_keys > 1].index)]

train_bin_df, val_bin_df = train_test_split(
    bin_df,
    test_size=0.2,
    random_state=SEED,
    stratify=bin_df["stratify_key"],
)

overlap = set(train_bin_df["sentence"]) & set(val_bin_df["sentence"])
print(f"[BINARY ZERO-SHOT] Sentence overlap : {len(overlap)}  (must be 0)")
print(f"[BINARY ZERO-SHOT] Train support: {len(train_bin_df)} | Val: {len(val_bin_df)}")
print("[BINARY ZERO-SHOT] Train label dist:\n", train_bin_df["label"].value_counts())
print("[BINARY ZERO-SHOT] Val label dist:\n", val_bin_df["label"].value_counts())


# ============================================================
# BLOCK 11 — BINARY EMBEDDINGS + PROTOTYPES
# ============================================================
train_bin_emb = embed_texts(
    train_bin_df["sentence"].tolist(), tokenizer, encoder,
    batch_size=16, max_len=128, device=DEVICE
)
val_bin_emb = embed_texts(
    val_bin_df["sentence"].tolist(), tokenizer, encoder,
    batch_size=16, max_len=128, device=DEVICE
)

bin_num_labels = 2
bin_prototypes = build_prototypes(
    train_bin_emb,
    train_bin_df["label"].values,
    num_classes=bin_num_labels,
)

bin_y_pred, bin_similarities = predict_with_prototypes(val_bin_emb, bin_prototypes)
bin_y_true = val_bin_df["label"].values

print("Binary zero-shot predictions ready.")


# ============================================================
# BLOCK 12 — BINARY EVALUATION
# ============================================================
acc, precision, recall, f1 = compute_metrics_from_preds(bin_y_true, bin_y_pred)

print("\n" + "=" * 55)
print("BINARY ZERO-SHOT EVALUATION")
print("=" * 55)
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")

print("\n[BINARY ZERO-SHOT] Prediction distribution:")
print(dict(zip(*[x.tolist() for x in np.unique(bin_y_pred, return_counts=True)])))

print("\n[BINARY ZERO-SHOT] Confusion matrix:")
print(confusion_matrix(bin_y_true, bin_y_pred))

print("\n[BINARY ZERO-SHOT] Classification report:")
print(classification_report(
    bin_y_true,
    bin_y_pred,
    target_names=["No Distortion", "Distorted"],
    digits=4,
))


# ============================================================
# BLOCK 13 — BINARY ROC-AUC
# ============================================================
bin_scores_tensor = torch.tensor(bin_similarities, dtype=torch.float)
bin_probs = torch.softmax(bin_scores_tensor, dim=1).numpy()

fpr_bin, tpr_bin, _ = roc_curve(bin_y_true, bin_probs[:, 1])
roc_auc_bin = auc(fpr_bin, tpr_bin)

print(f"\n[BINARY ZERO-SHOT] ROC-AUC: {roc_auc_bin:.4f}")


# ============================================================
# BLOCK 14 — MULTICLASS ZERO-SHOT PREP
# ============================================================
dist_df = df[df["label"] == 1].copy()
dist_df = dist_df[
    ~dist_df["distortion"].str.lower().isin(["no distortion", "none", ""])
].copy()

print("[MULTICLASS ZERO-SHOT] All class counts:")
print(dist_df["distortion"].value_counts())

distortion_classes = sorted(dist_df["distortion"].unique().tolist())
dist2id = {d: i for i, d in enumerate(distortion_classes)}
id2dist = {i: d for d, i in dist2id.items()}
dist_df["dist_label"] = dist_df["distortion"].map(dist2id)

assert dist_df["dist_label"].isnull().sum() == 0, "Null labels found!"

print(f"\n[MULTICLASS ZERO-SHOT] Num classes : {len(distortion_classes)}")
print(f"[MULTICLASS ZERO-SHOT] Classes     : {distortion_classes}")

dist_df["stratify_key"] = (
    dist_df["dist_label"].astype(str) + "_" + dist_df["language"].astype(str)
)
valid_keys = dist_df["stratify_key"].value_counts()
dist_df = dist_df[dist_df["stratify_key"].isin(valid_keys[valid_keys > 1].index)]

train_dist_df, val_dist_df = train_test_split(
    dist_df,
    test_size=0.2,
    random_state=SEED,
    stratify=dist_df["stratify_key"],
)

print(f"\n[MULTICLASS ZERO-SHOT] Train support: {len(train_dist_df)} | Val: {len(val_dist_df)}")
print("[MULTICLASS ZERO-SHOT] Train dist:\n", train_dist_df["distortion"].value_counts())
print("[MULTICLASS ZERO-SHOT] Val dist:\n", val_dist_df["distortion"].value_counts())


# ============================================================
# BLOCK 15 — MULTICLASS EMBEDDINGS + PROTOTYPES
# ============================================================
train_dist_emb = embed_texts(
    train_dist_df["sentence"].tolist(), tokenizer, encoder,
    batch_size=16, max_len=128, device=DEVICE
)
val_dist_emb = embed_texts(
    val_dist_df["sentence"].tolist(), tokenizer, encoder,
    batch_size=16, max_len=128, device=DEVICE
)

num_dist_labels = len(distortion_classes)
dist_prototypes = build_prototypes(
    train_dist_emb,
    train_dist_df["dist_label"].values,
    num_classes=num_dist_labels,
)

dist_y_pred, dist_similarities = predict_with_prototypes(val_dist_emb, dist_prototypes)
dist_y_true = val_dist_df["dist_label"].values

print("Multiclass zero-shot predictions ready.")


# ============================================================
# BLOCK 16 — MULTICLASS EVALUATION
# ============================================================
acc_d = accuracy_score(dist_y_true, dist_y_pred)

precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average="weighted", zero_division=0
)
precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average="macro", zero_division=0
)
precision_c, recall_c, f1_c, support_c = precision_recall_fscore_support(
    dist_y_true, dist_y_pred, average=None, zero_division=0
)

target_names_dist = [id2dist[i] for i in range(num_dist_labels)]

print("\n" + "=" * 55)
print("MULTICLASS ZERO-SHOT EVALUATION")
print("=" * 55)
print(f"Accuracy          : {acc_d:.4f}")
print(f"\nWeighted Average:")
print(f"  Precision : {precision_w:.4f}")
print(f"  Recall    : {recall_w:.4f}")
print(f"  F1        : {f1_w:.4f}")

print(f"\nMacro Average:")
print(f"  Precision : {precision_m:.4f}")
print(f"  Recall    : {recall_m:.4f}")
print(f"  F1        : {f1_m:.4f}")

print(f"\nPer-Class Results:")
print(f"{'Class':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 68)
for i, name in enumerate(target_names_dist):
    print(f"{name:<25} {precision_c[i]:>10.4f} {recall_c[i]:>10.4f} "
          f"{f1_c[i]:>10.4f} {int(support_c[i]):>10}")
print("-" * 68)

print("\n[MULTICLASS ZERO-SHOT] Prediction distribution:")
unique, counts = np.unique(dist_y_pred, return_counts=True)
print({id2dist[int(u)]: int(c) for u, c in zip(unique, counts)})

print("\n[MULTICLASS ZERO-SHOT] Confusion matrix:")
print(confusion_matrix(dist_y_true, dist_y_pred))

print("\n[MULTICLASS ZERO-SHOT] Full classification report:")
print(classification_report(
    dist_y_true,
    dist_y_pred,
    target_names=target_names_dist,
    digits=4,
))


# ============================================================
# BLOCK 17 — MULTICLASS ROC-AUC
# ============================================================
dist_scores_tensor = torch.tensor(dist_similarities, dtype=torch.float)
dist_probs = torch.softmax(dist_scores_tensor, dim=1).numpy()

y_true_bin = label_binarize(dist_y_true, classes=list(range(num_dist_labels)))

macro_auc_scores = []
for i in range(num_dist_labels):
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], dist_probs[:, i])
    auc_i = auc(fpr_i, tpr_i)
    macro_auc_scores.append(auc_i)

macro_auc = float(np.mean(macro_auc_scores))
print(f"\n[MULTICLASS ZERO-SHOT] Macro ROC-AUC: {macro_auc:.4f}")


# ============================================================
# BLOCK 18 — VISUALIZATION SETUP
# ============================================================
os.makedirs("./zero_shot_visualizations", exist_ok=True)
print("Visualization folder ready.")


# ============================================================
# BLOCK 19 — BINARY CONFUSION MATRIX
# ============================================================
cm_bin = confusion_matrix(bin_y_true, bin_y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_bin,
    display_labels=["No Distortion", "Distorted"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
for i in range(cm_bin.shape[0]):
    for j in range(cm_bin.shape[1]):
        pct = cm_bin[i, j] / cm_bin[i].sum() * 100 if cm_bin[i].sum() > 0 else 0
        ax.text(j, i + 0.3, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=10, color="black")
ax.set_title("Binary Zero-Shot — Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("./zero_shot_visualizations/binary_zero_shot_confusion_matrix.png", dpi=150)
plt.show()

print("Saved: binary_zero_shot_confusion_matrix.png")


# ============================================================
# BLOCK 20 — BINARY ROC-AUC CURVE
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_bin, tpr_bin, linewidth=2.5, label=f"ROC Curve (AUC = {roc_auc_bin:.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random Classifier")
ax.set_title("Binary Zero-Shot — ROC-AUC Curve", fontsize=14, fontweight="bold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.legend(fontsize=10, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./zero_shot_visualizations/binary_zero_shot_roc_auc.png", dpi=150)
plt.show()

print("Saved: binary_zero_shot_roc_auc.png")


# ============================================================
# BLOCK 21 — MULTICLASS CONFUSION MATRIX
# ============================================================
cm_dist = confusion_matrix(dist_y_true, dist_y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dist,
    display_labels=target_names_dist
)
disp.plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=45)
ax.set_title("Multiclass Zero-Shot — Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("./zero_shot_visualizations/multiclass_zero_shot_confusion_matrix.png", dpi=150)
plt.show()

print("Saved: multiclass_zero_shot_confusion_matrix.png")


# ============================================================
# BLOCK 22 — MULTICLASS ROC-AUC CURVES
# ============================================================
fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, num_dist_labels))

for i, (label, color) in enumerate(zip(target_names_dist, colors)):
    fpr_i, tpr_i, _ = roc_curve(y_true_bin[:, i], dist_probs[:, i])
    auc_i = auc(fpr_i, tpr_i)
    ax.plot(fpr_i, tpr_i, linewidth=2, color=color, label=f"{label} (AUC={auc_i:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Random")
ax.set_title("Multiclass Zero-Shot — ROC-AUC (One-vs-Rest)", fontsize=13, fontweight="bold")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./zero_shot_visualizations/multiclass_zero_shot_roc_auc.png", dpi=150)
plt.show()

print("Saved: multiclass_zero_shot_roc_auc.png")


# ============================================================
# BLOCK 23 — SAVE RESULTS
# ============================================================
results_summary = {
    "binary_zero_shot": {
        "accuracy": float(acc),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
        "f1_weighted": float(f1),
        "roc_auc": float(roc_auc_bin),
    },
    "multiclass_zero_shot": {
        "accuracy": float(acc_d),
        "precision_weighted": float(precision_w),
        "recall_weighted": float(recall_w),
        "f1_weighted": float(f1_w),
        "precision_macro": float(precision_m),
        "recall_macro": float(recall_m),
        "f1_macro": float(f1_m),
        "macro_roc_auc": float(macro_auc),
    },
    "distortion_classes": distortion_classes,
}

with open("zero_shot_results_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("Saved: zero_shot_results_summary.json")


# ============================================================
# BLOCK 24 — FINAL SUMMARY
# ============================================================
print("\n" + "=" * 55)
print("FINAL ZERO-SHOT SUMMARY")
print("=" * 55)
print("[BINARY ZERO-SHOT]")
print(f"Accuracy : {acc:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc_bin:.4f}")

print("\n[MULTICLASS ZERO-SHOT]")
print(f"Accuracy : {acc_d:.4f}")
print(f"Weighted F1 : {f1_w:.4f}")
print(f"Macro F1    : {f1_m:.4f}")
print(f"Macro ROC-AUC : {macro_auc:.4f}")

print("\nVisualizations saved to ./zero_shot_visualizations/")
print("=" * 55)